# NumCompute Quickstart Demo

This notebook demonstrates the NumCompute toolkit as an end-to-end scientific computing and lightweight ML workflow across different datasets.

We cover:
- CSV loading using `io.py`
- Missing value inspection and imputation
- Preprocessing using `preprocessing.py`
- Train-test split using `utils.py`
- Pipeline chaining using `pipeline.py`
- Estimator protocol using `models.py`
- Metrics evaluation using `metrics.py`
- Sorting, searching, top-k, quickselect, ranking, and percentiles
- Descriptive statistics and axis-wise behaviour
- Finite-difference gradient and Jacobian estimation
- Benchmarking loop-based vs vectorised implementations with timing and memory usage


## 1. Loading the required libraries

The notebook is stored inside the `demo/` folder, so we add the project root to `sys.path` before importing `numcompute`.


In [1]:
# Standard library imports
import os
import sys
import csv
import time

# Add project root to Python path so that numcompute can be imported from demo/
BASE_DIR = os.path.abspath("..")
sys.path.append(BASE_DIR)

# NumPy is allowed by the assignment
import numpy as np

# NumCompute modules
from numcompute.io import read_csv
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, ColumnTransformer, Imputer
from numcompute.sort_search import sort, topk, quickselect, binary_search
from numcompute.rank import rank, percentile
from numcompute.stats import mean, median, variance, std, minimum, maximum, histogram, quantiles, describe
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.models import ZeroRClassifier, ZeroRRegressor
from numcompute.metrics import accuracy, precision, recall, f1, mse, rmse, mad, mape, confusion_matrix, roc_curve, auc
from numcompute.utils import train_test_split
from numcompute.benchmarking import run_benchmarks

print("Imports completed successfully.")
print("Project root:", BASE_DIR)

Imports completed successfully.
Project root: d:\Programming for Aritificial Intelligence\programming_task_1-main


## 2. Environment notes

These details support reproducibility of the demo and benchmarking results.


In [2]:
print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("Current working directory:", os.getcwd())

Python version: 3.10.19
NumPy version: 2.2.6
Current working directory: d:\Programming for Aritificial Intelligence\programming_task_1-main\demo


## 3. Loading datasets using `io.py`

We load all datasets using the custom `read_csv()` function from `io.py`.  
The first row contains headers, so we remove it after loading. Header removal is done here, not inside `io.py`, so the I/O module remains dataset-agnostic.

Dataset roles:
- **Iris**: multi-class classification
- **WineQT**: regression using quality score as target
- **Sleep**: multi-class classification using sleep disorder type as target


In [3]:
datasets = {
    "Iris": os.path.join(BASE_DIR, "data", "Iris.csv"),        # Multi-class classification
    "WineQT": os.path.join(BASE_DIR, "data", "WineQT.csv"),   # Regression
    "Sleep": os.path.join(BASE_DIR, "data", "sleep_data.csv") # Multi-class classification
}

data_dict = {}

for name, path in datasets.items():
    # Load using custom io.py reader
    raw_data = read_csv(path)

    # Remove header row at usage level
    data = raw_data[1:]
    data_dict[name] = data

    print(f"\n{name} Dataset:")
    print("Shape:", data.shape)
    print("Sample first 5 rows and first 5 columns:")
    print(data[:5, :5])

print("\nDataset Summary:")
for name, data in data_dict.items():
    print(f"{name}: rows={data.shape[0]}, cols={data.shape[1]}")


Iris Dataset:
Shape: (150, 6)
Sample first 5 rows and first 5 columns:
[['1' '5.1' '3.5' '1.4' '0.2']
 ['2' '4.9' '3.0' '1.4' '0.2']
 ['3' '4.7' '3.2' '1.3' '0.2']
 ['4' '4.6' '3.1' '1.5' '0.2']
 ['5' '5.0' '3.6' '1.4' '0.2']]

WineQT Dataset:
Shape: (1143, 13)
Sample first 5 rows and first 5 columns:
[['7.4' '0.7' '0.0' '1.9' '0.076']
 ['7.8' '0.88' '0.0' '2.6' '0.098']
 ['7.8' '0.76' '0.04' '2.3' '0.092']
 ['11.2' '0.28' '0.56' '1.9' '0.075']
 ['7.4' '0.7' '0.0' '1.9' '0.076']]

Sleep Dataset:
Shape: (374, 13)
Sample first 5 rows and first 5 columns:
[['1' 'Male' '27' 'Software Engineer' '6.1']
 ['2' 'Male' '28' 'Doctor' '6.2']
 ['3' 'Male' '28' 'Doctor' '6.2']
 ['4' 'Male' '28' 'Sales Representative' '5.9']
 ['5' 'Male' '28' 'Sales Representative' '5.9']]

Dataset Summary:
Iris: rows=150, cols=6
WineQT: rows=1143, cols=13
Sleep: rows=374, cols=13


## 4. Missing value inspection

This section checks missing-like entries before preprocessing. Because `read_csv()` returns an object array, we check common missing markers such as empty strings, `nan`, and actual `NaN` values.


In [4]:
def count_missing_like(data):
    # Convert to object array for safe comparison across mixed types
    arr = np.asarray(data, dtype=object)

    missing_count = 0

    # Count blank values and common string representations of missing values
    missing_count += np.sum(arr == "")
    missing_count += np.sum(arr == "nan")
    missing_count += np.sum(arr == "NaN")
    missing_count += np.sum(arr == "None")

    # Count actual floating NaN values if present
    for value in arr.ravel():
        try:
            if isinstance(value, float) and np.isnan(value):
                missing_count += 1
        except TypeError:
            pass

    return int(missing_count)

for name, data in data_dict.items():
    print(f"{name} missing-like values:", count_missing_like(data))

Iris missing-like values: 0
WineQT missing-like values: 0
Sleep missing-like values: 234


## 5. Train-test split for the Iris classification task

For the main classification demonstration, we use the Iris dataset.  
The features are numeric, while the target labels are categorical strings. Since `metrics.py` expects numeric labels, we encode the target separately using NumPy.


In [5]:
# Select Iris dataset
iris = data_dict["Iris"]

# Features and target
X_iris = iris[:, :-1].astype(float)
y_iris_text = iris[:, -1]

# Encode categorical target labels into numeric labels for metrics.py
classes, y_iris = np.unique(y_iris_text, return_inverse=True)

print("Class mapping:")
for idx, label in enumerate(classes):
    print(idx, "=", label)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_iris,
    y_iris,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Class mapping:
0 = Iris-setosa
1 = Iris-versicolor
2 = Iris-virginica
Train shape: (120, 5)
Test shape: (30, 5)


## 6. Preprocessing components: Imputer and StandardScaler

We first demonstrate individual preprocessing components.  
The imputer replaces missing values, and the scaler standardizes features to mean 0 and standard deviation 1.


In [6]:
# Create a small copy with an artificial missing value to demonstrate imputation
X_missing_demo = X_train.copy()
X_missing_demo[0, 0] = np.nan

print("Missing values before imputation:", np.isnan(X_missing_demo).sum())

# Handle missing values using mean imputation
imputer = Imputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_missing_demo)
X_test_imp = imputer.transform(X_test)

print("Missing values after imputation:", np.isnan(X_train_imp).sum())

# Apply standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("Sample after imputation and scaling:")
print(X_train_scaled[:5])

Missing values before imputation: 1
Missing values after imputation: 0
Sample after imputation and scaling:
[[ 3.34166850e-16 -2.70269635e-01 -8.03439971e-01  2.67483980e-01
   1.21525474e-01]
 [-4.20699373e-01  5.80016857e-01  5.57681862e-01  5.53563104e-01
   5.15662145e-01]
 [ 5.19896689e-01 -1.48800136e-01 -3.49732693e-01  2.67483980e-01
   1.21525474e-01]
 [ 6.84501000e-01  5.80016857e-01 -3.49732693e-01  1.06850553e+00
   7.78419927e-01]
 [-5.14758980e-01  1.30883385e+00  1.03974584e-01  6.67994753e-01
   3.84283255e-01]]


## 7. Preprocessing components: OneHotEncoder and ColumnTransformer

The `OneHotEncoder` handles categorical feature columns.  
The `ColumnTransformer` demonstrates mixed preprocessing: numeric columns are imputed and scaled, while categorical columns are one-hot encoded.


In [7]:
# OneHotEncoder demo with simple categorical features
X_cat = np.array([
    ["Red", "Small"],
    ["Blue", "Medium"],
    ["Red", "Medium"],
    ["Green", "Small"]
], dtype=object)

encoder = OneHotEncoder()
X_cat_encoded = encoder.fit_transform(X_cat)

print("Original categorical data:")
print(X_cat)
print("One-hot encoded output:")
print(X_cat_encoded)

# ColumnTransformer demo with mixed numeric and categorical data
X_mixed = np.array([
    [25, "Male"],
    [np.nan, "Female"],
    [35, "Female"],
    [40, "Male"]
], dtype=object)

ct = ColumnTransformer(num_cols=[0], cat_cols=[1])
X_mixed_out = ct.fit_transform(X_mixed)

print("Mixed data before ColumnTransformer:")
print(X_mixed)
print("Mixed data after ColumnTransformer:")
print(X_mixed_out)

Original categorical data:
[['Red' 'Small']
 ['Blue' 'Medium']
 ['Red' 'Medium']
 ['Green' 'Small']]
One-hot encoded output:
[[0 0 1 0 1]
 [1 0 0 1 0]
 [0 0 1 1 0]
 [0 1 0 0 1]]
Mixed data before ColumnTransformer:
[[25 'Male']
 [nan 'Female']
 [35 'Female']
 [40 'Male']]
Mixed data after ColumnTransformer:
[[-1.5430335  0.         1.       ]
 [ 0.         1.         0.       ]
 [ 0.3086067  1.         0.       ]
 [ 1.2344268  0.         1.       ]]


## 8. Sorting, searching, top-k, quickselect, ranking, and percentiles

This section demonstrates `sort_search.py` and `rank.py`.

We include a small array with duplicate values to show tie handling in ranking.


In [8]:
sample = X_train_scaled[:, 0]

print("Sorted sample values:")
print(sort(sample)[:10])

print("Top 5 values:")
print(topk(sample, 5))

sorted_sample = np.sort(sample)
idx, found = binary_search(sorted_sample, sorted_sample[0])
print("Binary search result:")
print("Index:", idx, "Found:", found)

quick_arr = np.array([9, 1, 5, 3, 7])
print("Quickselect example array:", quick_arr)
print("3rd smallest value:", quickselect(quick_arr, 2))

rank_demo = np.array([10, 20, 20, 30, 10])
print("Rank demo array with ties:", rank_demo)
print("Average ranks:", rank(rank_demo, method="average"))
print("Dense ranks:", rank(rank_demo, method="dense"))
print("Ordinal ranks:", rank(rank_demo, method="ordinal"))

print("50th percentile of sample:", percentile(sample, 50))

Sorted sample values:
[-1.71401896 -1.61995935 -1.59644445 -1.57292955 -1.54941465 -1.52589975
 -1.50238485 -1.47886994 -1.45535504 -1.43184014]
Top 5 values:
[1.67212687 1.69564177 1.71915667 1.74267157 1.76618647]
Binary search result:
Index: 0 Found: True
Quickselect example array: [9 1 5 3 7]
3rd smallest value: 5.0
Rank demo array with ties: [10 20 20 30 10]
Average ranks: [1.5 3.5 3.5 5.  1.5]
Dense ranks: [1. 2. 2. 3. 1.]
Ordinal ranks: [1. 3. 4. 5. 2.]
50th percentile of sample: -0.09149075145717259


## 9. Statistical analysis using `stats.py`

We compute descriptive statistics, histogram, quantiles, and axis-wise statistics.


In [9]:
print("Sample values first 10:")
print(sample[:10])

print("Basic Statistics:")
print("Mean:", mean(sample))
print("Median:", median(sample))
print("Standard deviation:", std(sample))

# Variance is shown if implemented in stats.py
if "variance" in globals():
    print("Variance:", variance(sample))

print("Minimum:", minimum(sample))
print("Maximum:", maximum(sample))

hist, bin_edges = histogram(sample, bins=5)
print("Histogram counts:")
print(hist)
print("Histogram bin edges:")
print(bin_edges)

print("Quantiles [25, 50, 75]:")
print(quantiles(sample, [25, 50, 75]))

print("Column-wise mean (axis=0):")
print(mean(X_train_scaled, axis=0))

print("Row-wise mean first 5 values (axis=1):")
print(mean(X_train_scaled, axis=1)[:5])

print("Describe summary:")
print(describe(sample))

Sample values first 10:
[ 3.34166850e-16 -4.20699373e-01  5.19896689e-01  6.84501000e-01
 -5.14758980e-01 -1.43184014e+00  9.43164917e-01 -1.07911662e+00
  1.34291824e+00 -1.61995935e+00]
Basic Statistics:
Mean: 2.3684757858670006e-16
Median: -0.09149075145717259
Standard deviation: 1.0
Variance: 1.0000000000000002
Minimum: -1.7140189592646158
Maximum: 1.7661864719745377
Histogram counts:
[23 28 23 25 21]
Histogram bin edges:
[-1.71401896 -1.01797787 -0.32193679  0.3741043   1.07014539  1.76618647]
Quantiles [25, 50, 75]:
[-0.80281652 -0.09149075  0.83146913]
Column-wise mean (axis=0):
[ 2.44249065e-16  2.70154269e-16 -1.14723046e-15 -1.53580852e-15
 -3.51570624e-17]
Row-wise mean first 5 values (axis=1):
[-0.13694003  0.35724492  0.08207466  0.55234212  0.39006549]
Describe summary:
{'mean': np.float64(2.3684757858670006e-16), 'median': np.float64(-0.09149075145717259), 'variance': np.float64(1.0000000000000002), 'std': np.float64(1.0), 'min': np.float64(-1.7140189592646158), 'max': n

## 10. Gradient and Jacobian estimation using `optim.py`

Gradients and Jacobians are computed for mathematical functions at a specific point.  
They represent local derivatives, so they are demonstrated on scalar and vector functions rather than directly across a dataset.

We first verify gradients on simple functions with known derivatives.
We then demonstrate a parameterised loss function to show how gradients
can be used in optimisation workflows such as regression.


In [ ]:
# Scalar function
# -----------------------------
# f(x) = x1^2 + x2^2 + x3^2
# Expected gradient = [2*x1, 2*x2, 2*x3]
def scalar_function(x):
    return np.sum(x ** 2)

point = np.array([1.0, 2.0, 3.0])

gradient_central = grad(scalar_function, point, method="central")
gradient_forward = grad(scalar_function, point, method="forward")

print("Point:", point)
print("\n--- Gradient of scalar function ---")
print("Central difference gradient:", gradient_central)
print("Forward difference gradient:", gradient_forward)
print("Expected gradient:", 2 * point)

# -----------------------------
#  Vector function
# -----------------------------
# F(x) = [x1 + x2, x2 * x3]
# Since output is vector-valued, jacobian() is used.
def vector_function(x):
    return np.array([
        x[0] + x[1],
        x[1] * x[2]
    ])

jacobian_matrix = jacobian(vector_function, point, method="central")

print("\n--- Jacobian of vector function ---")
print(jacobian_matrix)

# -----------------------------
# Parameterised loss function
# -----------------------------
# Small regression-style example:
# prediction = w0*x + w1
# loss = mean squared error
X_small = np.array([1.0, 2.0, 3.0, 4.0])
y_small = np.array([3.0, 5.0, 7.0, 9.0])  # approximately y = 2x + 1

def mse_loss(params):
    w0, w1 = params
    predictions = w0 * X_small + w1
    return np.mean((y_small - predictions) ** 2)

params = np.array([0.5, 0.0])

loss_gradient = grad(mse_loss, params, method="central")

print("\n--- Gradient of parameterised MSE loss ---")
print("Parameters [w0, w1]:", params)
print("Loss:", mse_loss(params))
print("Gradient:", loss_gradient)

# One small gradient-descent style update
learning_rate = 0.1
updated_params = params - learning_rate * loss_gradient

print("Updated parameters after one step:", updated_params)
print("Updated loss:", mse_loss(updated_params))

Point: [1. 2. 3.]

--- Gradient of scalar function ---
Central difference gradient: [2. 4. 6.]
Forward difference gradient: [2.00001 4.00001 6.00001]
Expected gradient: [2. 4. 6.]

--- Jacobian of vector function ---
[[1. 1. 0.]
 [0. 3. 2.]]

--- Gradient of parameterised MSE loss ---
Parameters [w0, w1]: [0.5 0. ]
Loss: 25.375
Gradient: [-27.5  -9.5]
Updated parameters after one step: [3.25 0.95]
Updated loss: 11.408750000262433


## 11. Pipeline demonstration using Transformer and Estimator protocol

The pipeline chains a transformer and an estimator:

- `StandardScaler` follows the transformer API: `fit()` + `transform()`
- `ZeroRClassifier` follows the estimator API: `fit()` + `predict()`

This demonstrates the required pipeline protocol without implementing a complex model.


In [11]:
sleep = data_dict["Sleep"]

# Last column is treated as the target sleep disorder type
X = sleep[:, :-1]
y = sleep[:, -1]

# Convert categorical target labels into numeric labels for metrics.py
classes, y = np.unique(y, return_inverse=True)

print("Class mapping:")
for i, label in enumerate(classes):
    print(i, "=", label)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

# Build pipeline:
# ColumnTransformer automatically detects numeric and categorical feature columns.
classification_pipe = Pipeline([
    ("preprocess", ColumnTransformer()),
    ("model", ZeroRClassifier())
])

# Fit pipeline and predict
classification_pipe.fit(X_train, y_train)
y_pred = classification_pipe.predict(X_test)

print("\nAuto-detected numeric columns:")
print(classification_pipe.steps[0][1].num_cols)

print("\nAuto-detected categorical columns:")
print(classification_pipe.steps[0][1].cat_cols)

print("\nPredictions:")
print(y_pred[:10])

print("\nActual:")
print(y_test[:10])

Class mapping:
0 = Insomnia
1 = None
2 = Sleep Apnea

Auto-detected numeric columns:
[0, 2, 5, 6, 11]

Auto-detected categorical columns:
[1, 3, 4, 7, 8, 9, 10]

Predictions:
[1 1 1 1 1 1 1 1 1 1]

Actual:
[1 2 0 1 1 1 1 1 0 1]


## 12. Classification metrics using `metrics.py`

We evaluate the classification pipeline using accuracy and confusion matrix.


In [12]:
acc = accuracy(y_test, y_pred)
prec = precision(y_test, y_pred)
rec = recall(y_test, y_pred)
f1_score = f1(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1_score)
print("Confusion Matrix:")
print(cm)

# ROC curve + AUC for a simple one-vs-rest view (class 0)
y_true_bin = (y_test == 0).astype(int)
y_pred_bin = (y_pred == 0).astype(int)
fpr, tpr = roc_curve(y_true_bin, y_pred_bin)
roc_auc = auc(fpr, tpr)
print("ROC AUC (class 0 vs rest):", roc_auc)

Accuracy: 0.581081081081081
Precision: 0.7049180327868853
Recall: 1.0
F1: 0.8269230769230769
Confusion Matrix:
[[ 0 18  0]
 [ 0 43  0]
 [ 0 13  0]]
ROC AUC (class 0 vs rest): 0.5059523809523809


## 13. Regression pipeline and metrics using WineQT

We use WineQT as a regression task, where the target is the quality score.  
`ZeroRRegressor` predicts the mean target value and is used only to demonstrate the estimator protocol.


In [13]:
wine = data_dict["WineQT"]

# Last column is treated as target quality score
X_wine = wine[:, :-1].astype(float)
y_wine = wine[:, -1].astype(float)

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine,
    y_wine,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

regression_pipe = Pipeline([
    ("preprocess", StandardScaler()),
    ("model", ZeroRRegressor())
])

regression_pipe.fit(X_train_w, y_train_w)
y_pred_w = regression_pipe.predict(X_test_w)

print("Regression predictions:", y_pred_w[:10])
print("Actual values:", y_test_w[:10])

print("MSE:", mse(y_test_w, y_pred_w))
print("RMSE:", rmse(y_test_w, y_pred_w))
print("MAD/MAE:", mad(y_test_w, y_pred_w))
print("MAPE:", mape(y_test_w, y_pred_w))

Regression predictions: [812.4568306 812.4568306 812.4568306 812.4568306 812.4568306 812.4568306
 812.4568306 812.4568306 812.4568306 812.4568306]
Actual values: [1592.  760. 1271. 1336.  461.  771.  701.  146.  120.   94.]
MSE: 214912.8406757914
RMSE: 463.5869289311244
MAD/MAE: 399.9841913526987
MAPE: 734.4973442666565


## 14. Benchmarking vectorised implementations vs Python loops

The benchmarking module is reusable:

- It can be run as a standalone script: `python -m numcompute.benchmarking`
- It can also be imported and run from this notebook

For this demo, we run the benchmark without saving CSV output.

Memory usage is measured using Python's built-in `tracemalloc`, which tracks Python-level memory allocation. It does not represent total system RAM usage, but it provides a consistent comparison across implementations.


In [14]:
from numcompute.benchmarking import run_benchmarks

benchmark_datasets = {
    "Iris": os.path.join(BASE_DIR, "data", "Iris.csv"),
    "WineQT": os.path.join(BASE_DIR, "data", "WineQT.csv"),
    "Sleep": os.path.join(BASE_DIR, "data", "sleep_data.csv")
}

benchmark_results = run_benchmarks(
    benchmark_datasets,
    save_csv=False
)

Loading datasets...

Iris: shape=(150, 5)
WineQT: shape=(1143, 13)
Sleep: shape=(374, 5)

BENCHMARK RESULTS
------------------------------------------------------------------------------------------------------------------------
Dataset   Operation           Loop(s)   NumComp(s)  Loop(MB)  NumComp(MB) Speedup
------------------------------------------------------------------------------------------------------------------------
Iris      mean                0.000542  0.000025    0.0003    0.0140      21.36x
Iris      std                 0.000873  0.000028    0.0003    0.0142      31.64x
Iris      sort                0.000064  0.000008    0.0252    0.0084      7.96x
Iris      topk                0.000062  0.000006    0.0252    0.0113      10.11x
Iris      rank                0.020305  0.000049    0.1902    0.0602      413.55x
Iris      percentile          0.000070  0.000010    0.0252    0.0142      6.81x
Iris      pairwise_distances  0.031161  0.000183    0.3928    0.9107      169.81x
W